In [ ]:
#!/usr/bin/env python3
"""
Feature-selection workflow for best_bayesian_150.csv.

Outputs:
  correlation_matrix.csv
  correlation_filtered_features.csv
  mutual_information_scores.csv
  boruta_results.csv
  final_feature_ranking.csv
  final_selected_features.csv
and publication-quality plots in <output_dir>/plots.
"""

from __future__ import annotations

import argparse
import os
import sys
import tempfile
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "matplotlib_codex"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder


DEFAULT_INPUT_CANDIDATES = [
    Path("bayesiannetwork_150.csv"),
    Path("best_bayesian_150.csv"),
    Path("outputs/datasets/best_bayesian_150.csv"),
]

TARGET_CANDIDATES = [
    "Target_Class",
    "Diagnosis_Label",
    "Diagnosis",
    "target",
    "Target",
    "class",
    "Class",
    "label",
    "Label",
]

IDENTIFIER_HINTS = ("id", "subject_id", "sample_id", "record_id")


def parse_args(args_list: list[str] | None = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Run correlation filtering, mutual information, Boruta, and final feature ranking."
    )
    parser.add_argument(
        "--input",
        type=Path,
        default=None,
        help="Input CSV. Defaults to best_bayesian_150.csv or outputs/datasets/best_bayesian_150.csv.",
    )
    parser.add_argument(
        "--target",
        default=None,
        help="Target column. Defaults to Target_Class if present, otherwise a common target-like column.",
    )
    parser.add_argument(
        "--output-dir",
        type=Path,
        default=Path("feature_selection_outputs"),
        help="Directory for CSVs and plots.",
    )
    parser.add_argument("--correlation-threshold", type=float, default=0.90)
    parser.add_argument("--boruta-iterations", type=int, default=100)
    parser.add_argument("--random-state", type=int, default=42)
    # Use args_list if provided, otherwise default to empty list in notebooks to avoid kernel flag conflicts
    return parser.parse_args(args_list if args_list is not None else [])


def resolve_input(path: Path | None) -> Path:
    if path is not None:
        if not path.exists():
            raise FileNotFoundError(f"Input file not found: {path}")
        return path

    for candidate in DEFAULT_INPUT_CANDIDATES:
        if candidate.exists():
            return candidate

    choices = ", ".join(str(p) for p in DEFAULT_INPUT_CANDIDATES)
    raise FileNotFoundError(f"No input CSV found. Expected one of: {choices}")


def resolve_target(df: pd.DataFrame, target: str | None) -> str:
    if target is not None:
        if target not in df.columns:
            raise ValueError(f"Target column '{target}' not found in input CSV.")
        return target

    for candidate in TARGET_CANDIDATES:
        if candidate in df.columns:
            return candidate

    raise ValueError(
        "Could not infer target column. Pass --target explicitly, for example --target Target_Class."
    )


def likely_identifier_columns(df: pd.DataFrame, target: str) -> list[str]:
    identifiers: list[str] = []
    for col in df.columns:
        normalized = col.strip().lower().replace(" ", "_")
        if col == target:
            continue
        if normalized in IDENTIFIER_HINTS or normalized.endswith("_id"):
            identifiers.append(col)
    return identifiers


def prepare_feature_matrix(df: pd.DataFrame, features: list[str]) -> tuple[pd.DataFrame, np.ndarray]:
    X = df[features].copy()
    discrete = np.zeros(len(features), dtype=bool)

    for idx, col in enumerate(features):
        if pd.api.types.is_numeric_dtype(X[col]):
            X[col] = pd.to_numeric(X[col], errors="coerce")
            median = X[col].median()
            if not np.isfinite(median):
                median = 0.0
            X[col] = X[col].fillna(median)
        else:
            discrete[idx] = True
            values = X[[col]].astype("string").fillna("__missing__")
            encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
            X[col] = encoder.fit_transform(values).ravel()

    return X.astype(float), discrete


def prepare_target(y: pd.Series) -> tuple[np.ndarray, bool]:
    if pd.api.types.is_numeric_dtype(y) and y.nunique(dropna=True) > 10:
        numeric_y = pd.to_numeric(y, errors="coerce")
        median = numeric_y.median()
        if not np.isfinite(median):
            median = 0.0
        return numeric_y.fillna(median).to_numpy(), False

    encoder = LabelEncoder()
    labels = y.astype("string").fillna("__missing__")
    return encoder.fit_transform(labels), True


def correlation_filter(
    df: pd.DataFrame,
    features: list[str],
    threshold: float,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    numeric_features = [col for col in features if pd.api.types.is_numeric_dtype(df[col])]
    corr = df[numeric_features].corr(method="pearson") if numeric_features else pd.DataFrame()

    recommendations: dict[str, dict[str, object]] = {
        feature: {
            "feature": feature,
            "correlation_filter_status": "Keep",
            "max_abs_correlation": 0.0,
            "paired_feature": "",
            "correlation_value": np.nan,
            "recommendation": "Keep",
            "reason": "No Pearson correlation above threshold.",
        }
        for feature in features
    }

    to_remove: set[str] = set()
    if not corr.empty:
        mean_abs_corr = corr.abs().replace(1.0, np.nan).mean(skipna=True).fillna(0)
        for i, left in enumerate(corr.columns):
            for right in corr.columns[i + 1 :]:
                value = corr.loc[left, right]
                abs_value = abs(value)
                if abs_value > recommendations[left]["max_abs_correlation"]:
                    recommendations[left]["max_abs_correlation"] = abs_value
                    recommendations[left]["paired_feature"] = right
                    recommendations[left]["correlation_value"] = value
                if abs_value > recommendations[right]["max_abs_correlation"]:
                    recommendations[right]["max_abs_correlation"] = abs_value
                    recommendations[right]["paired_feature"] = left
                    recommendations[right]["correlation_value"] = value
                if abs_value >= threshold:
                    remove = left if mean_abs_corr[left] >= mean_abs_corr[right] else right
                    keep = right if remove == left else left
                    to_remove.add(remove)
                    recommendations[remove].update(
                        {
                            "correlation_filter_status": "Remove",
                            "recommendation": "Remove",
                            "reason": (
                                f"Highly correlated with '{keep}' "
                                f"(Pearson r={value:.3f}); higher mean absolute correlation."
                            ),
                        }
                    )

    for feature in to_remove:
        recommendations[feature]["correlation_filter_status"] = "Remove"

    rec_df = pd.DataFrame(recommendations.values()).sort_values(
        ["correlation_filter_status", "max_abs_correlation", "feature"],
        ascending=[True, False, True],
    )
    return corr, rec_df


def mutual_information(
    X: pd.DataFrame,
    y: np.ndarray,
    discrete_features: np.ndarray,
    classification: bool,
    random_state: int,
) -> pd.DataFrame:
    if classification:
        scores = mutual_info_classif(
            X, y, discrete_features=discrete_features, random_state=random_state
        )
    else:
        scores = mutual_info_regression(
            X, y, discrete_features=discrete_features, random_state=random_state
        )

    result = pd.DataFrame({"feature": X.columns, "mutual_information": scores})
    result["mi_rank"] = result["mutual_information"].rank(ascending=False, method="min").astype(int)
    return result.sort_values(["mi_rank", "feature"])


def boruta_shadow_selection(
    X: pd.DataFrame,
    y: np.ndarray,
    classification: bool,
    iterations: int,
    random_state: int,
) -> pd.DataFrame:
    rng = np.random.default_rng(random_state)
    hit_counts = pd.Series(0, index=X.columns, dtype=float)
    importances = pd.Series(0.0, index=X.columns)

    estimator_cls = RandomForestClassifier if classification else RandomForestRegressor
    estimator_kwargs = {
        "n_estimators": 500,
        "max_depth": None,
        "n_jobs": -1,
        "random_state": random_state,
    }
    if classification:
        estimator_kwargs["class_weight"] = "balanced"

    X_values = X.to_numpy()
    for iteration in range(iterations):
        shadow = X_values.copy()
        for col_idx in range(shadow.shape[1]):
            rng.shuffle(shadow[:, col_idx])

        combined = np.hstack([X_values, shadow])
        estimator = estimator_cls(**{**estimator_kwargs, "random_state": random_state + iteration})
        estimator.fit(combined, y)

        real_importance = pd.Series(estimator.feature_importances_[: X.shape[1]], index=X.columns)
        shadow_importance = estimator.feature_importances_[X.shape[1] :]
        shadow_threshold = np.max(shadow_importance)
        hit_counts += (real_importance > shadow_threshold).astype(float)
        importances += real_importance

    hit_rate = hit_counts / iterations
    mean_importance = importances / iterations

    status = np.where(hit_rate >= 0.80, "Confirmed", np.where(hit_rate <= 0.20, "Rejected", "Tentative"))
    result = pd.DataFrame(
        {
            "feature": X.columns,
            "boruta_status": status,
            "boruta_hit_rate": hit_rate.values,
            "boruta_mean_importance": mean_importance.values,
        }
    )
    result["boruta_rank"] = result["boruta_mean_importance"].rank(ascending=False, method="min").astype(int)
    status_order = {"Confirmed": 0, "Tentative": 1, "Rejected": 2}
    result["_status_order"] = result["boruta_status"].map(status_order)
    return result.sort_values(["_status_order", "boruta_rank", "feature"]).drop(columns="_status_order")


def build_final_ranking(
    features: list[str],
    corr_rec: pd.DataFrame,
    mi: pd.DataFrame,
    boruta: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    final = pd.DataFrame({"feature": features})
    final = final.merge(mi, on="feature", how="left")
    final = final.merge(boruta, on="feature", how="left")
    final = final.merge(
        corr_rec[
            [
                "feature",
                "correlation_filter_status",
                "max_abs_correlation",
                "paired_feature",
                "correlation_value",
                "reason",
            ]
        ],
        on="feature",
        how="left",
    )

    status_points = {"Confirmed": 3.0, "Tentative": 1.5, "Rejected": 0.0}
    corr_points = {"Keep": 1.0, "Remove": 0.0}
    max_mi = final["mutual_information"].max()
    if not np.isfinite(max_mi) or max_mi <= 0:
        final["mi_scaled"] = 0.0
    else:
        final["mi_scaled" ] = final["mutual_information"] / max_mi

    final["boruta_points"] = final["boruta_status"].map(status_points).fillna(0.0)
    final["correlation_points"] = final["correlation_filter_status"].map(corr_points).fillna(0.0)
    final["final_score"] = (
        0.50 * final["mi_scaled"]
        + 0.35 * (final["boruta_points"] / 3.0)
        + 0.15 * final["correlation_points"]
    )
    final["selected"] = (
        final["correlation_filter_status"].eq("Keep")
        & final["boruta_status"].isin(["Confirmed", "Tentative"])
    )
    if not final["selected"].any():
        final["selected"] = (
            final["correlation_filter_status"].eq("Keep")
            & (final["mi_rank"] <= min(10, max(1, len(final))))
        )

    final = final.sort_values(["final_score", "mutual_information", "feature"], ascending=[False, False, True])
    final["final_rank"] = np.arange(1, len(final) + 1)
    final = final[
        [
            "final_rank",
            "feature",
            "selected",
            "final_score",
            "mutual_information",
            "mi_rank",
            "boruta_status",
            "boruta_hit_rate",
            "boruta_mean_importance",
            "boruta_rank",
            "correlation_filter_status",
            "max_abs_correlation",
            "paired_feature",
            "correlation_value",
            "reason",
        ]
    ]
    selected = final.loc[final["selected"], ["final_rank", "feature", "final_score"]].copy()
    return final, selected


def save_plots(
    corr: pd.DataFrame,
    mi: pd.DataFrame,
    boruta: pd.DataFrame,
    final: pd.DataFrame,
    plot_dir: Path,
    threshold: float,
) -> None:
    plot_dir.mkdir(parents=True, exist_ok=True)
    sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
    plt.rcParams.update(
        {
            "figure.dpi": 150,
            "savefig.dpi": 300,
            "font.family": "DejaVu Sans",
            "axes.titleweight": "bold",
            "axes.labelsize": 11,
            "axes.titlesize": 13,
            "xtick.labelsize": 9,
            "ytick.labelsize": 9,
        }
    )

    if not corr.empty:
        fig_width = max(8, min(18, 0.55 * len(corr.columns)))
        fig_height = max(7, min(18, 0.55 * len(corr.columns)))
        fig, ax = plt.subplots(figsize=(fig_width, fig_height), constrained_layout=True)
        sns.heatmap(
            corr,
            cmap="vlag",
            center=0,
            vmin=-1,
            vmax=1,
            square=True,
            linewidths=0.25,
            cbar_kws={"label": "Pearson r", "shrink": 0.75},
            ax=ax,
        )
        ax.set_title(f"Pearson Correlation Heatmap (threshold = {threshold:.2f})")
        ax.set_xlabel("")
        ax.set_ylabel("")
        fig.savefig(plot_dir / "correlation_heatmap.png", bbox_inches="tight")
        fig.savefig(plot_dir / "correlation_heatmap.pdf", bbox_inches="tight")
        plt.close(fig)

    mi_plot = mi.sort_values("mutual_information", ascending=True)
    fig_height = max(5, 0.38 * len(mi_plot))
    fig, ax = plt.subplots(figsize=(9, fig_height), constrained_layout=True)
    sns.barplot(data=mi_plot, x="mutual_information", y="feature", color="#2f6f8f", ax=ax)
    ax.set_title("Mutual Information Feature Ranking")
    ax.set_xlabel("Mutual information")
    ax.set_ylabel("")
    fig.savefig(plot_dir / "mutual_information_ranking.png", bbox_inches="tight")
    fig.savefig(plot_dir / "mutual_information_ranking.pdf", bbox_inches="tight")
    plt.close(fig)

    boruta_plot = boruta.sort_values("boruta_mean_importance", ascending=True)
    palette = {"Confirmed": "#2f8f5b", "Tentative": "#d6a33b", "Rejected": "#9b3d3d"}
    fig_height = max(5, 0.38 * len(boruta_plot))
    fig, ax = plt.subplots(figsize=(9, fig_height), constrained_layout=True)
    sns.barplot(
        data=boruta_plot,
        x="boruta_mean_importance",
        y="feature",
        hue="boruta_status",
        dodge=False,
        palette=palette,
        ax=ax,
    )
    ax.set_title("Boruta Random Forest Feature Status")
    ax.set_xlabel("Mean Random Forest importance")
    ax.set_ylabel("")
    ax.legend(title="Status", loc="lower right", frameon=True)
    fig.savefig(plot_dir / "boruta_feature_status.png", bbox_inches="tight")
    fig.savefig(plot_dir / "boruta_feature_status.pdf", bbox_inches="tight")
    plt.close(fig)

    final_plot = final.sort_values("final_score", ascending=True)
    fig_height = max(5, 0.38 * len(final_plot))
    fig, ax = plt.subplots(figsize=(9, fig_height), constrained_layout=True)
    colors = np.where(final_plot["selected"], "#2f8f5b", "#6f7785")
    ax.barh(final_plot["feature"], final_plot["final_score"], color=colors)
    ax.set_title("Final Recommended Feature Ranking")
    ax.set_xlabel("Final score")
    ax.set_ylabel("")
    fig.savefig(plot_dir / "final_feature_ranking.png", bbox_inches="tight")
    fig.savefig(plot_dir / "final_feature_ranking.pdf", bbox_inches="tight")
    plt.close(fig)


def main() -> None:
    # Pass an empty list to parse_args if running in a notebook context
    args = parse_args(args_list=[])
    input_path = resolve_input(args.input)
    output_dir = args.output_dir
    output_dir.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(input_path)
    target = resolve_target(df, args.target)
    identifiers = likely_identifier_columns(df, target)
    features = [col for col in df.columns if col not in identifiers + [target]]
    if not features:
        raise ValueError("No feature columns remain after excluding target and identifier columns.")

    y, classification = prepare_target(df[target])
    X, discrete_features = prepare_feature_matrix(df, features)

    corr, corr_rec = correlation_filter(df, features, args.correlation_threshold)
    mi = mutual_information(X, y, discrete_features, classification, args.random_state)
    boruta = boruta_shadow_selection(
        X,
        y,
        classification,
        iterations=args.boruta_iterations,
        random_state=args.random_state,
    )
    final, selected = build_final_ranking(features, corr_rec, mi, boruta)

    corr.to_csv(output_dir / "correlation_matrix.csv")
    corr_rec.to_csv(output_dir / "correlation_filtered_features.csv", index=False)
    mi.to_csv(output_dir / "mutual_information_scores.csv", index=False)
    boruta.to_csv(output_dir / "boruta_results.csv", index=False)
    final.to_csv(output_dir / "final_feature_ranking.csv", index=False)
    selected.to_csv(output_dir / "final_selected_features.csv", index=False)

    save_plots(corr, mi, boruta, final, output_dir / "plots", args.correlation_threshold)

    print(f"Input: {input_path}")
    print(f"Target: {target}")
    print(f"Excluded identifiers: {identifiers if identifiers else 'none'}")
    print(f"Features ranked: {len(features)}")
    print(f"Selected features: {len(selected)}")
    print(f"Outputs saved to: {output_dir}")


if __name__ == "__main__":
    main()

In [ ]:
from IPython.display import Image, display
import pandas as pd
import os

# 1. Visualize the final_feature_ranking plot
ranking_plot_path = 'feature_selection_outputs/plots/final_feature_ranking.png'
if os.path.exists(ranking_plot_path):
    display(Image(filename=ranking_plot_path))
else:
    print('Ranking plot not found at:', ranking_plot_path)

In [ ]:
# 2. Show descriptive statistics for the selected features
selected_features_df = pd.read_csv('feature_selection_outputs/final_selected_features.csv')
selected_feature_names = selected_features_df['feature'].tolist()

# Load original data to get statistics
df = pd.read_csv('bayesiannetwork_150.csv')
stats = df[selected_feature_names].describe()
display(stats)

In [ ]:
# 3. Export all generated output into a zip file and download
import shutil
from google.colab import files

# Zip the directory
shutil.make_archive('feature_selection_results', 'zip', 'feature_selection_outputs')

# Trigger download
files.download('feature_selection_results.zip')